# 3. Contributing to LaStBeRu

Three ways to contribute, from the easiest to the most involved:

1. **Report a problem** (a wrong coordinate, redshift or classification, a duplicated system): open an issue in [CosmoObs/slcomp](https://github.com/CosmoObs/slcomp/issues).
2. **Add a catalog** whose lenses are not yet in the compilation.
3. **Improve the pipeline**: cross-matches with new surveys, new cutouts, tests.

Contributions 2 and 3 go to the development repository [`CosmoObs/slcomp-dev`](https://github.com/CosmoObs/slcomp-dev), private for now (ask for access at fisica.renan@gmail.com).

## The development repository

```
slcomp-dev
├── 01_Code/src.py                  -> shared helpers (coordinates, matching, cutouts)
├── 02_Data
│   ├── 01_Processed_Data/          -> one folder per paper (extraction)
│   │   └── Diehl_et_al._2009/
│   │       ├── dataset/            -> the table as published (never edited)
│   │       ├── Diehl_et_al._2009.ipynb
│   │       └── Diehl_et_al._2009.csv
│   ├── 02_Database/                -> PostgreSQL schema, loader, reviews, exports
│   ├── 03_Cross_Matches/           -> photometric and spectroscopic matches
│   └── 04_Consolidated_Data/       -> one value per quantity
└── 03_Cutouts/                     -> cutout download and image processing
```

---
## Adding a catalog, step by step

Suppose Negrello et al. (2016), the Herschel-ATLAS lens candidates seen in notebook 1, were **not yet** in LaStBeRu:

1. save the original files (VizieR `ReadMe` + `table4.dat`) in `02_Data/01_Processed_Data/Negrello_et_al._2016/dataset/`;
2. write `Negrello_et_al._2016.ipynb`, which reads `dataset/` and writes `Negrello_et_al._2016.csv` following the conventions of notebook 1;
3. check which systems are new and which are already known;
4. open a pull request.

### 1–2. Read and standardise

In [1]:
import io

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier

table = Vizier(row_limit=-1).get_catalogs("J/MNRAS/465/3558/table4")[0].to_pandas()
print(table.columns.tolist())
table.head()

['Field', 'HATLAS', 'n_HATLAS', 'F250', 'F350', 'F500', 'Rel', 'Sep', 'rmag', 'zSDSS', 'f_zSDSS', 'n_zSDSS', 'zSPIRE', 'f_zSPIRE', 'n_zSPIRE', 'Lrank', 'l_mu', 'mu', 'Simbad', '_RA', '_DE']


,Field,HATLAS,n_HATLAS,F250,F350,F500,Rel,Sep,rmag,zSDSS,...,n_zSDSS,zSPIRE,f_zSPIRE,n_zSPIRE,Lrank,l_mu,mu,Simbad,_RA,_DE
0,GAMA-9h field (53.4 deg2),J083051.0+013225,(B13),248.500000,305.299988,269.100006,0.958,1.81,21.600000,0.6260,...,,3.6340,s,a,A,,6.9,Simbad,127.71250,1.54000
1,GAMA-9h field (53.4 deg2),J085358.9+015537,(B13),396.399994,367.899994,228.199997,NaN,NaN,NaN,NaN,...,,2.0925,s,b,A,,15.3,Simbad,133.49448,1.92652
2,GAMA-9h field (53.4 deg2),J091043.0-000322,(SDP.11),420.799988,370.500000,221.399994,NaN,NaN,NaN,0.7930,...,,1.7860,s,,A,,10.9,Simbad,137.67958,-0.05583
3,GAMA-9h field (53.4 deg2),J091043.0-000322,(SDP.11),NaN,NaN,NaN,0.027,4.39,20.400000,0.4570,...,,NaN,,,,,NaN,Simbad,137.67958,-0.05583
4,GAMA-9h field (53.4 deg2),J090302.9-014127,(SDP.17),354.100006,338.799988,220.000000,0.988,0.83,21.700001,0.9435,...,,2.3049,s,,A,>,4.9,Simbad,135.76208,-1.69083


Read the [VizieR `ReadMe`](https://cdsarc.cds.unistra.fr/ftp/J/MNRAS/465/3558/ReadMe) before mapping the columns to the dictionary names:

* **Table 4 has no position columns.** The published position is the IAU name (`JHHMMSS.s+DDMMSS`, the Herschel/SPIRE position truncated to 0.1 s in RA and 1″ in Dec). `_RA` and `_DE` were added by VizieR from SIMBAD and are not published data: we derive the positions from the names and use SIMBAD only for comparison.
* `zSDSS` is the redshift of the optical counterpart (the lens candidate) and `zSPIRE` that of the submillimetre source; `f_zSDSS` and `f_zSPIRE` mark each one as spectroscopic (`s`) or photometric (`p`).

In [2]:
names = table["HATLAS"].str.strip()  # e.g. J083051.0+013225
coords = SkyCoord(
    [f"{n[1:3]}:{n[3:5]}:{n[5:9]} {n[9:12]}:{n[12:14]}:{n[14:16]}" for n in names],
    unit=(u.hourangle, u.deg),
)

simbad = SkyCoord(table["_RA"].values * u.deg, table["_DE"].values * u.deg)
offset = coords.separation(simbad).arcsec
print(
    f"published (name) vs SIMBAD positions: {(offset > 0.1).sum()} rows differ by > 0.1'', max {offset.max():.1f}''"
)


def split_by_flag(z, flag, kind):
    """Put each redshift in Spec-z_<kind> or Photo-z_<kind> according to its flag."""
    return {
        f"Spec-z_{kind}": z.where(flag.str.strip() == "s"),
        f"Photo-z_{kind}": z.where(flag.str.strip() == "p"),
    }


new = pd.DataFrame(
    {
        "ID": "HATLAS" + table["HATLAS"].str.strip(),
        "RA": coords.ra.deg.round(6),
        "DEC": coords.dec.deg.round(6),
        "Coordinates": [
            c.replace(" ", "") for c in coords.to_string("hmsdms", sep=":", precision=2)
        ],
        "mag_r": table["rmag"],
        **split_by_flag(table["zSDSS"], table["f_zSDSS"], "L"),
        **split_by_flag(table["zSPIRE"], table["f_zSPIRE"], "S"),
        "Grade": table["Lrank"].str.strip().replace("", np.nan),
        "Reference": "Negrello et al. (2016) - arXiv:1611.03922",
    }
)
print("rows with a repeated ID:", new.ID.duplicated(keep=False).sum())
new = new.drop_duplicates().reset_index(drop=True)
new.insert(0, "record_id", new.index + 1)
new.head()

published (name) vs SIMBAD positions: 26 rows differ by > 0.1'', max 3.7''
rows with a repeated ID: 4


,record_id,ID,RA,DEC,Coordinates,mag_r,Spec-z_L,Photo-z_L,Spec-z_S,Photo-z_S,Grade,Reference
0,1,HATLASJ083051.0+013225,127.712500,1.540278,08:30:51.00+01:32:25.00,21.600000,0.6260,NaN,3.6340,NaN,A,Negrello et al. (2016) - arXiv:1611.03922
1,2,HATLASJ085358.9+015537,133.495417,1.926944,08:53:58.90+01:55:37.00,NaN,NaN,NaN,2.0925,NaN,A,Negrello et al. (2016) - arXiv:1611.03922
2,3,HATLASJ091043.0-000322,137.679167,-0.056111,09:10:43.00-00:03:22.00,NaN,0.7930,NaN,1.7860,NaN,A,Negrello et al. (2016) - arXiv:1611.03922
3,4,HATLASJ091043.0-000322,137.679167,-0.056111,09:10:43.00-00:03:22.00,20.400000,NaN,0.457,NaN,NaN,NaN,Negrello et al. (2016) - arXiv:1611.03922
4,5,HATLASJ090302.9-014127,135.762083,-1.690833,09:03:02.90-01:41:27.00,21.700001,0.9435,NaN,2.3049,NaN,A,Negrello et al. (2016) - arXiv:1611.03922


> Two Herschel sources appear twice, with different `mag_r` and `Photo-z_L`: the paper lists **two possible optical counterparts** for each. Dropping rows by `ID` would lose a lens candidate; this is why the key is the row (`record_id`), not the name.

Before committing, a few automatic checks catch most mistakes:

In [3]:
import re

REFERENCE = re.compile(r"^.+ \(\d{4}\) - ((arXiv|doi|adsabs):|https?://|www)\S+$")


def check_processed_catalog(df):
    """Return a list of problems found in a processed literature catalog."""
    problems = []
    if list(df.record_id) != list(range(1, len(df) + 1)):
        problems.append("record_id must be 1..N, unique and sequential")
    if not df.RA.between(0, 360, inclusive="left").all():
        problems.append("RA outside [0, 360)")
    if not df.DEC.between(-90, 90).all():
        problems.append("DEC outside [-90, 90]")
    bad_refs = df.loc[~df.Reference.str.match(REFERENCE), "Reference"].unique()
    if len(bad_refs):
        problems.append(
            f"Reference not in 'Author et al. (Year) - arXiv:...' (or doi:, adsabs:, URL) format: {bad_refs[:3]}"
        )
    if df.drop(columns="record_id").duplicated().any():
        problems.append("duplicated rows")
    for col in df.columns:
        if (col.startswith(("Spec-z", "Photo-z", "mag_", "theta_E", "velDisp"))) and df[
            col
        ].dtype == object:
            problems.append(f"{col} should be numeric")
    return problems or ["OK"]


check_processed_catalog(new)

['OK']

### 3. New or already known?

Cross-match the new catalog with the systems already in the compilation. A small separation makes a pair only a **candidate association**: the radius is a choice (≈2″ for galaxy-scale lenses with optical positions), and each candidate must be confirmed with the images and the papers.

In [4]:
from minio import Minio

# Public, read-only credentials (see the slcomp README)
client = Minio(
    "l5s5a0sibv6w.share.zrok.io",
    access_key="slcomp",
    secret_key="slcomp@data",
    secure=True,
)


def read_csv_object(name, **kwargs):
    """Read a CSV stored in the slcomp bucket into a DataFrame."""
    data = client.get_object("slcomp", name).data
    return pd.read_csv(io.BytesIO(data), low_memory=False, **kwargs)


consolidated = read_csv_object("Data/Consolidated_Data.csv")
known = SkyCoord(consolidated.RA.values * u.deg, consolidated.DEC.values * u.deg)

idx, sep, _ = SkyCoord(
    new.RA.values * u.deg, new.DEC.values * u.deg
).match_to_catalog_sky(known)
new["nearest_JNAME"] = consolidated.JNAME.values[idx]
new["sep_arcsec"] = sep.arcsec.round(2)

print("candidate association (< 2''):", (new.sep_arcsec < 2).sum())
print("ambiguous (2''-10''):", new.sep_arcsec.between(2, 10).sum())
print("no counterpart within 10'':", (new.sep_arcsec > 10).sum())
new.sort_values("sep_arcsec").head(10)

candidate association (< 2''): 81
ambiguous (2''-10''): 1
no counterpart within 10'': 0


,record_id,ID,RA,DEC,Coordinates,mag_r,Spec-z_L,Photo-z_L,Spec-z_S,Photo-z_S,Grade,Reference,nearest_JNAME,sep_arcsec
22,23,HATLASJ115820.1-013752,179.583750,-1.631111,11:58:20.10-01:37:52.00,NaN,NaN,NaN,2.1911,NaN,C,Negrello et al. (2016) - arXiv:1611.03922,J115820.1-013752.0,0.00
19,20,HATLASJ120127.6-014043,180.365000,-1.678611,12:01:27.60-01:40:43.00,NaN,NaN,NaN,NaN,3.80,C,Negrello et al. (2016) - arXiv:1611.03922,J120127.6-014043.0,0.00
50,51,HATLASJ130118.0+253708,195.325000,25.618889,13:01:18.00+25:37:08.00,NaN,NaN,NaN,NaN,4.08,C,Negrello et al. (2016) - arXiv:1611.03922,J130118.0+253708.0,0.00
40,41,HATLASJ133413.8+260457,203.557500,26.082500,13:34:13.80+26:04:57.00,NaN,NaN,NaN,NaN,2.63,C,Negrello et al. (2016) - arXiv:1611.03922,J133413.8+260457.0,0.00
74,75,HATLASJ013951.9-321446,24.966250,-32.246111,01:39:51.90-32:14:46.00,NaN,NaN,NaN,NaN,2.73,C,Negrello et al. (2016) - arXiv:1611.03922,J013951.9-321446.0,0.00
13,14,HATLASJ121334.9-020323,183.395417,-2.056389,12:13:34.90-02:03:23.00,18.700001,0.190,NaN,NaN,1.89,B,Negrello et al. (2016) - arXiv:1611.03922,J121334.9-020323.0,0.01
0,1,HATLASJ083051.0+013225,127.712500,1.540278,08:30:51.00+01:32:25.00,21.600000,0.626,NaN,3.6340,NaN,A,Negrello et al. (2016) - arXiv:1611.03922,J083051.0+013225.0,0.01
11,12,HATLASJ114637.9-001132,176.657917,-0.192222,11:46:37.90-00:11:32.00,21.299999,NaN,0.470,NaN,NaN,NaN,Negrello et al. (2016) - arXiv:1611.03922,J114637.9-001132.0,0.01
14,15,HATLASJ121301.5-004922,183.256250,-0.822778,12:13:01.50-00:49:22.00,21.600000,NaN,0.191,NaN,2.35,B,Negrello et al. (2016) - arXiv:1611.03922,J121301.5-004922.0,0.01
27,28,HATLASJ144556.1-004853,221.483750,-0.814722,14:45:56.10-00:48:53.00,NaN,NaN,NaN,NaN,2.51,C,Negrello et al. (2016) - arXiv:1611.03922,J144556.1-004853.0,0.01


Negrello et al. (2016) is already in LaStBeRu, so almost every row has a candidate association: a sanity check of the method. Confirming them is harder here: the positions are submillimetre detections (Herschel beam ~18–36″, names truncated to ~1″), and the optical counterpart can be up to ~5″ away (column `Sep`). Close pairs may be different objects, and wider pairs the same system. Decisions taken after inspecting the cutouts and the paper are recorded in `02_Data/02_Database/manual_reviews.csv` (notebook 4).

### 4. Git workflow

```bash
git clone git@github.com:CosmoObs/slcomp-dev.git
cd slcomp-dev
git switch -c add-negrello-2016            # one branch per contribution

# ... add the folder, run the notebook ...

ruff check --fix . && ruff format .         # lint and format (scripts and notebooks)
git add 02_Data/01_Processed_Data/Negrello_et_al._2016
git commit -m "Negrello 2016: add Herschel-ATLAS lens candidates"
git push -u origin add-negrello-2016
gh pr create --base dev                     # or open the PR on GitHub
```

Commit messages follow the pattern `<Source or component>: <what changed>`, e.g. `SuGOHI: correct redshift column for 145957-005522`.

### Reporting a correction

A good report allows anyone to check the problem without asking again:

| Field | Example |
|---|---|
| System | `J084259.0+362109.0` |
| Source and record | `Moustakas_et_al._2012`, record 1234 (or the paper and table row) |
| What is wrong | $z_S = z_L = 0.282$ |
| Evidence | Table 2 of the original paper gives $z_S = ...$; link or page |

---
## Find bugs!

A compilation of 75 heterogeneous sources has errors, and finding them is a real contribution. Some sanity checks on the consolidated catalog:

In [5]:
checks = {
    "source not behind the lens (z_S <= z_L)": consolidated.z_S <= consolidated.z_L,
    "galaxy lens with sigma > 500 km/s": (consolidated.Lens_Type == "GALAXY")
    & (consolidated.velDisp > 500),
    "galaxy lens with theta_E > 5''": (consolidated.Lens_Type == "GALAXY")
    & (consolidated.theta_E > 5),
    "velocity dispersion equal to 0 (sentinel)": consolidated.velDisp == 0,
    "lens redshift ~ source redshift (|z_S - z_L| < 0.05)": (
        consolidated.z_S - consolidated.z_L
    ).abs()
    < 0.05,
}
for name, mask in checks.items():
    print(f"{mask.sum():5d}  {name}")

    1  source not behind the lens (z_S <= z_L)
  167  galaxy lens with sigma > 500 km/s
    6  galaxy lens with theta_E > 5''
  261  velocity dispersion equal to 0 (sentinel)
   70  lens redshift ~ source redshift (|z_S - z_L| < 0.05)


In [6]:
consolidated[checks["lens redshift ~ source redshift (|z_S - z_L| < 0.05)"]][
    ["JNAME", "z_L", "z_LRef", "z_S", "z_SRef", "velDisp", "Source_Type"]
].head(10)

,JNAME,z_L,z_LRef,z_S,z_SRef,velDisp,Source_Type
1913,J005618.9+152532.0,0.070794,SDSS DR17,0.077625,Talbot et al. (2022) - arXiv:2206.09902,79.262880,NaN
5359,J022259.5-025827.2,0.570000,HSC Wide DR3,0.570000,SuGOHI Candidate List (2021) - www-utap.phys.s...,NaN,GALAXY
9432,J072940.0+410420.3,0.101025,SDSS DR17,0.118067,Talbot et al. (2022) - arXiv:2206.09902,194.332980,NaN
9501,J074352.6+245743.7,2.159544,SDSS DR17,2.165000,Moustakas et al. (2012) - adsabs:2012hst..prop...,0.000000,QUASAR
9532,J074809.3+433526.5,0.031138,SDSS DR17,0.050311,Talbot et al. (2022) - arXiv:2206.09902,92.456820,NaN
9580,J075519.9+140042.0,0.844891,SDSS DR17,0.845000,Moustakas et al. (2012) - adsabs:2012hst..prop...,0.000000,QUASAR
9582,J075533.7+502447.1,0.022046,SDSS DR17,0.062595,Talbot et al. (2022) - arXiv:2206.09902,44.536995,NaN
9621,J080004.1+232616.3,0.029218,SDSS DR17,0.068483,Talbot et al. (2022) - arXiv:2206.09902,156.089430,NaN
9673,J080623.7+200631.9,1.537252,SDSS DR17,1.537300,Moustakas et al. (2012) - adsabs:2012hst..prop...,0.000000,QUASAR
9693,J080924.4+261722.1,0.071520,SDSS DR17,0.109603,Talbot et al. (2022) - arXiv:2206.09902,114.924120,NaN


These 70 systems are not all the same problem: only some have a quasar source. Treat each explanation as a **hypothesis** and look for evidence, starting with the SDSS spectrum used for $z_L$. In `Spectroscopic_Match.csv`, `z` is the SDSS best fit and `Spec-z_L` the adopted value (`z_noqso`, notebook 1):

In [7]:
spectroscopic = read_csv_object("Data/Spectroscopic_Match.csv", dtype=object)
# Rows matched in several surveys join their values with " § "; SDSS always comes first
sdss = spectroscopic[spectroscopic.Catalog_Name.str.startswith("SDSS DR17")].copy()
for col in ["z", "Spec-z_L", "sourcetype", "Survey_ID"]:
    sdss[col] = sdss[col].str.split(" § ").str[0]
sdss = sdss[["JNAME", "z", "Spec-z_L", "sourcetype", "Survey_ID"]]

close = consolidated[checks["lens redshift ~ source redshift (|z_S - z_L| < 0.05)"]]
investigate = close[["JNAME", "z_L", "z_LRef", "z_S", "Source_Type"]].merge(
    sdss, on="JNAME", how="left"
)
investigate["spectrum"] = (
    "https://skyserver.sdss.org/dr18/VisualTools/explore/summary?sid="
    + investigate.Survey_ID
)
print(investigate.Source_Type.value_counts(dropna=False).to_string())
investigate.head(12)

Source_Type
QUASAR    30
NaN       28
GALAXY    12


,JNAME,z_L,z_LRef,z_S,Source_Type,z,Spec-z_L,sourcetype,Survey_ID,spectrum
0,J005618.9+152532.0,0.070794,SDSS DR17,0.077625,NaN,0.070794076,0.070794076,GALAXY,474098174454163456,https://skyserver.sdss.org/dr18/VisualTools/ex...
1,J022259.5-025827.2,0.570000,HSC Wide DR3,0.570000,GALAXY,NaN,NaN,NaN,NaN,NaN
2,J072940.0+410420.3,0.101025,SDSS DR17,0.118067,NaN,0.10102478,0.10102478,GALAXY,1952400374442780672,https://skyserver.sdss.org/dr18/VisualTools/ex...
3,J074352.6+245743.7,2.159544,SDSS DR17,2.165000,QUASAR,2.159544,2.159544,QSO,965034522573826048,https://skyserver.sdss.org/dr18/VisualTools/ex...
4,J074809.3+433526.5,0.031138,SDSS DR17,0.050311,NaN,0.031137768,0.031137768,GALAXY,488732950171510784,https://skyserver.sdss.org/dr18/VisualTools/ex...
5,J075519.9+140042.0,0.844891,SDSS DR17,0.845000,QUASAR,0.84456193,0.84489095,QSO,5067805591734605824,https://skyserver.sdss.org/dr18/VisualTools/ex...
6,J075533.7+502447.1,0.022046,SDSS DR17,0.062595,NaN,0.022045875,0.022045875,GALAXY,2104360033142728704,https://skyserver.sdss.org/dr18/VisualTools/ex...
7,J080004.1+232616.3,0.029218,SDSS DR17,0.068483,NaN,0.029217534,0.029217534,BRIGHTGAL,5027221247660152832,https://skyserver.sdss.org/dr18/VisualTools/ex...
8,J080623.7+200631.9,1.537252,SDSS DR17,1.537300,QUASAR,1.5372523,1.5372523,QSO,2164139655509862400,https://skyserver.sdss.org/dr18/VisualTools/ex...
9,J080924.4+261722.1,0.071520,SDSS DR17,0.109603,NaN,0.071519986,0.071519986,GALAXY,1356876558307977216,https://skyserver.sdss.org/dr18/VisualTools/ex...


Hypotheses to test, case by case:

* **The spectrum is of a quasar image** (target type `QSO`, $z \approx z_S$, $\sigma = 0$): open the spectrum link, check its class and emission lines, and compare the fibre position with the lens and the images in a cutout.
* **The spectrum is of the lens, but the source redshift is wrong or very close** (small $z_L$, $\Delta z \sim 0.01$–$0.05$, typical of spectroscopic searches): check the original paper; a close pair can be real.
* **Transcription error** (`z_L = z_S` exactly, both from the literature): go back to the table of the original paper.

Only once the cause is known can we choose the fix: a correction to a processed table, or a rule in the pipeline (e.g. in `02_Spectroscopic_Match.ipynb`).

### Exercise – the systems with $\sigma = 0$

The checks above found 261 systems with $\sigma = 0$, and only some of them have $z_L \approx z_S$. In notebook 2 there were also systems with $\sigma = 0$ and $z_L$ far below $z_S$ (e.g. `J141210.2+520423.3`: $z_L = 0.047$, $z_S = 2.952$).

1. Build the `investigate` table for the systems with `velDisp == 0` and `z_LRef == "SDSS DR17"`.
2. Compare `z`, `Spec-z_L` and `z_S`. What do you find?
3. Is the problem in a processed table or in a pipeline rule? Write the report, or the fix, you would propose.

In [ ]:
# Your checks here